# 04 - Simulation Interfaces & Multiformat Exporters
### poropack: 3D Porous Media & Digital Rock Physics Generator

This notebook demonstrates:
1. **Simulation File Exporters**: 3D TIFF stacks, ParaView VTK (`.vti` & `.vtp`), columnar Apache Parquet, and CSV.
2. **End-to-End MRST Integration**: Exporting pre-formatted `.mat` files for direct loading and simulation in the MATLAB Reservoir Simulation Toolbox (MRST).
3. **Complete MRST MATLAB Workflow**: Out-of-the-box script template complying with MRST `startup` and `mrstModule add ...` rules.


In [1]:
import os
import numpy as np
import poropack as pp

# Generate a sample compacted sandstone digital rock
pack = pp.RSAGenerator(
    box_size=(300.0, 300.0, 300.0),
    psd=pp.LogNormalPSD(d50=30.0, sigma_phi=0.25),
    random_state=42,
).generate(target_porosity=0.55)

grid = pack.rasterize(voxel_size=3.0)
compacted = pp.apply_compaction(grid, vertical_strain=0.10)
cemented = pp.apply_cementation(compacted, cement_fraction=0.08)

poro = cemented.porosity()
perm_mD = pp.kozeny_carman(cemented) / 9.869233e-16
print(f'Sample Rock Ready: Porosity = {poro:.3f}, Permeability = {perm_mD:.1f} mD')


Sample Rock Ready: Porosity = 0.414, Permeability = 1445433396445979.2 mD


## 1. Exporting to Industry Standard Formats

- **3D TIFF Stack**: Universal standard for ImageJ, Fiji, Avizo, Dragonfly, and PerGeos.
- **VTK ImageData (`.vti`)**: Volume rendering in ParaView.
- **VTK PolyData (`.vtp`)**: 3D sphere glyphs and contact topology in ParaView.
- **Apache Parquet**: Ultra-fast, compressed columnar storage of grain centroids, radii, and contact metadata.
- **CSV**: Standard tabular coordinate export.


In [2]:
# 1. Export 3D TIFF stack
pp.export_tiff(cemented, 'rock_sample.tif')

# 2. Export ParaView VTK ImageData
pp.export_vtk_vti(cemented, 'rock_volume.vti')

# 3. Export Lagrangian GrainPack to Apache Parquet
pp.export_parquet(pack, 'grains.parquet')

# 4. Export Lagrangian GrainPack to CSV
pp.export_csv(pack, 'grains.csv')

# 5. Export MATLAB MRST .mat struct
pp.export_mrst_mat(cemented, 'mrst_rock.mat', permeability_mD=perm_mD)

print('Exported Files:')
for fname in ['rock_sample.tif', 'rock_volume.vti', 'grains.parquet', 'grains.csv', 'mrst_rock.mat']:
    sz_kb = os.path.getsize(fname) / 1024.0
    print(f'  {fname:20s}: {sz_kb:8.1f} KB')
    os.remove(fname)  # Clean up temporary demonstration files


Exported Files:
  rock_sample.tif     :    893.6 KB
  rock_volume.vti     :    128.0 KB
  grains.parquet      :     46.8 KB
  grains.csv          :     88.3 KB
  mrst_rock.mat       :    156.1 KB


## 2. Seamless MATLAB / MRST Workflow

When `pp.export_mrst_mat(grid, 'mrst_rock.mat')` is called, it outputs a MATLAB struct containing:
- `G`: Cartesian grid geometry (`cartDims`, `cells.num`, `faces.num`, physical bounding dimensions).
- `rock`: Petrophysical property struct with `rock.poro` (voxel occupancy) and `rock.perm` (in mD).

### Out-of-the-Box MATLAB Script:
```matlab
% In MATLAB terminal:
clear; clc;
startup;  % Run MRST startup in pwd to initialize paths
mrstModule add incomp mrst-gui  % Load incompressible flow solver and 3D GUI

% Load poropack digital rock struct
data = load('mrst_rock.mat');
G = cartGrid(data.G.cartDims, data.G.dimensions);
G = computeGeometry(G);
rock = data.rock;

% Inspect rock properties
disp(['Average Porosity: ', num2str(data.rock.mean_poro)]);
disp(['Average Permeability: ', num2str(data.rock.mean_perm_md), ' mD']);

% Setup Dirichlet pressure boundaries (1 bar drop across X)
bc = pside([], G, 'XMin', 1.0*barsa);
bc = pside(bc, G, 'XMax', 0.0*barsa);

% Initialize fluid system
fluid = initSingleFluid('mu', 1.0*centi*poise, 'rho', 1000*kilogram/meter^3);

% Solve incompressible single-phase Darcy flow
state = incompTPFA(initResSol(G, 0.0), G, computeTrans(G, rock), fluid, 'bc', bc);

% Visualize pressure and flux in interactive 3D viewer
plotCellData(G, state.pressure / barsa, 'EdgeColor', 'none');
title('Darcy Pressure Distribution (bar) in poropack Digital Rock');
colorbar; view(3); axis tight;
```
